In [36]:
# @title Configurations

from torchvision import transforms
import torch

# --- CONFIGURATIONS: SET PATHS, DEVICE, MODEL AND HYPERPARAMETERS ---

CONFIGURATION = {
	"PATH_DRIVE": "/content/drive",
	"PATH_EXPORT": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/",
	"PATH_FILE": "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/SPair-71k.tar.gz",

	# "PTH_PATH": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/dinov2_vitb14_reg4_pretrain.pth",
  # "PTH_PATH": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth",
  "PTH_PATH": "/content/drive/MyDrive/AML-Semantic-Correspondence/weights/sam_vit_b_01ec64.pth",

  "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
  "MODEL_VERSION": "sam",                                        # "dinov2", "dinov3", "sam"

 # DATASET SPAIR-71k

  "PATH_TEST_SPAIR71K": "/content/SPair-71k/PairAnnotation/test",
  "PATH_VAL_SPAIR71K": "/content/SPair-71k/PairAnnotation/val",
  "PATH_TRAIN_SPAIR71K": "/content/SPair-71k/PairAnnotation/trn",

  "ALL_TEST_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/test.txt",
  "ALL_TRAIN_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/trn.txt",
  "ALL_VAL_PATH_SPAIR71K": "/content/SPair-71k/Layout/small/val.txt",

  "IMAGE_FOLDER_NAME_SPAIR71K": "/content/SPair-71k/JPEGImages",

  # FOR INFERENCE

  "IMAGE_SIZE": 1024,           # 518 for dinov2, 512 for dinov3 and 1024 for sam (TO ASK IF IT'S CORRECT)
	"ALPHA": [0.05, 0.1, 0.2]
}

MODEL = None

# --- RESIZE IMAGE TO STANDARD MODEL DIMENSIONS, CONVERT IT INTO A TENSOR AND NORMALIZE IT ---

PREPROCESS = transforms.Compose([
    transforms.Resize((CONFIGURATION["IMAGE_SIZE"], CONFIGURATION["IMAGE_SIZE"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [37]:
# @title Datasets

from torch.utils.data import Dataset, DataLoader
# from Configurations import CONFIGURATION
import os, json, numpy as np

# --- DATASET CLASSES ---
# --- COLLECT ALL JSON FILE BASED ---
# --- WHEN REQUIRED, OPEN THE NEXT FILE AND TAKE IN A DICTIONARY ALL YOU NEED ---

class SPair71kDataset(Dataset):
    def __init__(self, pair_path, source_path, folder_path):
        self.image_path = source_path
        self.folder_path = folder_path
        file = open(pair_path, "r")
        self.pair_files = file.readlines()                # TAKE ALL JSON FILES
        file.close()
        return

    def __len__(self):
        return len(self.pair_files)

    def __getitem__(self, idx):
        file_name = self.pair_files[idx].strip()
        category = file_name.split(".json")[0].split(":")[1]
        json_path = os.path.join(self.image_path, file_name + ".json")

        file = open(json_path, "r")
        annotation = json.load(file)             # LOAD INFORMATIONS
        file.close()

        src_path = os.path.join(self.folder_path, category, annotation["src_imname"])
        trg_path = os.path.join(self.folder_path, category, annotation["trg_imname"])
        ids = [int(el) for el in annotation["kps_ids"]]

        return {
            "src_path": src_path,
            "trg_path": trg_path,
            "src_kps": np.array(annotation["src_kps"]),
            "trg_kps": np.array(annotation["trg_kps"]),
            "kps_ids": np.array(ids),
            "trg_bndbox": np.array(annotation["trg_bndbox"]),        # DICT of BATCHES (SIZE=1)
        }

def custom_collate_fn(batch):
    collated_batch = {}
    keys = batch[0].keys()
    for key in keys:
        collated_batch[key] = [d[key] for d in batch]              # COLLATE FOR DATALOADER
    return collated_batch

# --- GENERATE DATALOADER FROM REQUIRED OPERATION --

def Loader(index, batch_size):
    if index == 0:                                         # FOR INFERENCE WITH SPAIR
        dataset=SPair71kDataset(CONFIGURATION["ALL_TEST_PATH_SPAIR71K"], CONFIGURATION["PATH_TEST_SPAIR71K"],
                                CONFIGURATION["IMAGE_FOLDER_NAME_SPAIR71K"])

    if batch_size>1:
        loader=DataLoader(dataset, batch_size=batch_size, collate_fn=custom_collate_fn)
    else:
        loader=DataLoader(dataset, batch_size=1)

    return loader

In [38]:
# @title Utils

import math
# from Configurations import CONFIGURATION, PREPROCESS, MODEL
from PIL import Image
from matplotlib import pyplot as plt
import torch, cv2
import torch.nn.functional as F

# --- HELPER FUNCTIONS ---
# --- LOAD AND PREPROCESS IMAGE ---
# --- RESHAPE AND NORMALIZE FEATURE MAPS ---
# --- FINALLY RETURN THE EXTRACTED FEATURE MAPS AND THE ORIGINAL DIMENSIONS ---

def get_descriptors(img_path, grad):
    img = Image.open(img_path).convert("RGB")
    (w, h) = img.size
    input_tensor = PREPROCESS(img).unsqueeze(0).to(CONFIGURATION["DEVICE"])

    with torch.set_grad_enabled(grad):
        if CONFIGURATION["MODEL_VERSION"] == "dinov2":
            x = MODEL.get_intermediate_layers(input_tensor, n=1)[0]   # [B, N, D]
            (B, N, D) = x.shape
            H = int(math.sqrt(N))
            x = x.reshape(B, H, H, D)                # RESHAPE

        elif CONFIGURATION["MODEL_VERSION"] == "dinov3":
            x = MODEL.forward_features(input_tensor)["x_norm_patchtokens"]
            (B, N, D) = x.shape
            H = int(math.sqrt(N))
            x = x.reshape(B, H, H, D)               # RESHAPE

        elif CONFIGURATION["MODEL_VERSION"] == "sam":
            x = MODEL.image_encoder(input_tensor).permute(0, 2, 3, 1)

    x = F.normalize(x, dim=-1)
    return (x, w, h)

# --- GET PREDICTIONS ---
# --- FOR EACH SRC_KPS RESCALE IT, USE COSINE SIMILARITY METRIC AND COMPUTE THE PREDICTION WITH SIMPLE ARGMAX  ---

def get_predictions(batch, index=0):
    src_kps = batch["src_kps"][index]

    (feat_src, sw, sh) = get_descriptors(batch["src_path"][index], False)
    (feat_trg, tw, th) = get_descriptors(batch["trg_path"][index], False)             # DESCRIPTORS

    (_, Hf, Wf, D) = feat_trg.shape
    trg_flat = feat_trg[0].reshape(Hf * Wf, D)

    pred_kps = []

    for i in range(src_kps.shape[0]):
        sx = int(src_kps[i, 0] * Wf / sw)
        sy = int(src_kps[i, 1] * Hf / sh)
        sx = torch.clamp(torch.tensor(sx, device=CONFIGURATION["DEVICE"]), 0, Wf - 1)              # RESIZE
        sy = torch.clamp(torch.tensor(sy, device=CONFIGURATION["DEVICE"]), 0, Hf - 1)

        src_desc = feat_src[0, sy, sx, :]
        sim = torch.matmul(trg_flat, src_desc)            # COSINE SIMILARITY

        best_idx = sim.argmax()
        pred_xy = torch.tensor([best_idx % Wf, best_idx // Wf], device=CONFIGURATION["DEVICE"])           # PREDICTION

        pred_x = (pred_xy[0] + 0.5) * (tw / Wf)
        pred_y = (pred_xy[1] + 0.5) * (th / Hf)
        pred_kps.append(torch.stack([pred_x, pred_y]))

    return torch.stack(pred_kps)

# --- VISUALIZE RESULTS AND COMPARE CORRECT AND PREDICTED KEYPOINTS ON TARGET IMAGE. ---

def visualize_keypoints(src_path, trg_path, src_kps, pred_kps, trg_kps):
    src_img = cv2.imread(src_path)[:, :, ::-1]
    trg_img = cv2.imread(trg_path)[:, :, ::-1]

    (_, axes) = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(src_img)
    axes[0].scatter(src_kps[:,0], src_kps[:,1], c="r", s=40, label="src_kps")
    axes[0].set_title("Source Image")

    axes[1].imshow(trg_img)
    axes[1].scatter(pred_kps[:,0], pred_kps[:,1], c="b", s=40, label="pred_kps")
    axes[1].scatter(trg_kps[:,0], trg_kps[:,1], c="g", s=40, marker="X", label="gt_kps")
    axes[1].set_title("Target Image")

    plt.legend()
    plt.show()

In [39]:
# @title Inference

# from Configurations import CONFIGURATION
# from Utils import get_predictions, visualize_keypoints
from tqdm.auto import tqdm
import torch

# --- EVALUATION FUNCTION ---
# --- FOR EACH PAIR: LOAD SOURCE AND TARGET IMAGES ---
# --- EXTRACT DESCRIPTORS AND RESCALE COORDINATES ---
# --- USE COSINE SIMILARITY IN ORDER TO FIND THE CORRESPONDING POINT IN THE TARGET IMAGE. ---
# --- FINALLY, RETURN THE CORRECT GENERATED KEYPOINTS (USING THEIR DISTANCE FROM THE ORIGINAL ONE) RATIO. ---
# --- OPTIONALLY: VISUALIZE RESULTS. ---

def run_evaluation(loader, split_desc, visualize=False):
    total_correct = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}
    total_images = 0
    total_correct_keypoints = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}
    total_keypoints = 0

    for batch in tqdm(loader, desc="Evaluating " + split_desc + " PCK metrics"):
        trg_kps = torch.as_tensor(batch["trg_kps"][0]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)
        trg_bndbox = torch.as_tensor(batch["trg_bndbox"][0]).to(device=CONFIGURATION["DEVICE"], dtype=torch.float32)

        pred_kps = get_predictions(batch)

        max_dim = max(trg_bndbox[2]-trg_bndbox[0], trg_bndbox[3]-trg_bndbox[1])
        total_correct_image = {alpha: 0 for alpha in CONFIGURATION["ALPHA"]}    # KEYPOINTS CORRECTLY CLASSIFIED
        total_points_image = 0                                      # FOR THE CURRENT IMAGE
        total_images += 1

        for i in range(0, len(batch["kps_ids"][0])):
            total_keypoints += 1

            dist = torch.norm(pred_kps[i] - trg_kps[i]).item()            # DISTANCE METRIC
            total_points_image += 1

            for alpha in CONFIGURATION["ALPHA"]:

                if dist <= alpha * max_dim:                     # PREDICTION IS CORRECT?
                    total_correct[alpha] += 1
                    total_correct_image[alpha] += 1

        # PRINT PER IMAGE

        for alpha in CONFIGURATION["ALPHA"]:
            total_correct_keypoints[alpha] += 100 * total_correct_image[alpha]
            total_correct_image[alpha] = round(100 * total_correct_image[alpha] / total_points_image, 2)
            total_correct[alpha] += total_correct_image[alpha]

        if visualize:
            visualize_keypoints(batch["src_path"][0], batch["trg_path"][0], batch["src_kps"][0], pred_kps,trg_kps)

    print("PCK@t results per image:")
    for alpha in CONFIGURATION["ALPHA"]:
        total_correct[alpha] = round(total_correct[alpha] / total_images, 2)
        total_correct_keypoints[alpha] = round(total_correct_keypoints[alpha] / total_keypoints, 2)
        print("PCK@" + str(alpha) + ": " + str(total_correct[alpha]) + "%")

    print()
    print("PCK@t results per keypoint:")
    for alpha in CONFIGURATION["ALPHA"]:
        print("PCK@" + str(alpha) + ": " + str(total_correct_keypoints[alpha]) + "%")

    return

In [ ]:
# @title Main

!pip install torchmetrics                # ONLY FOR FIRST EXECUTION

from google.colab import drive
# from Inference import Inference_step
# from Configurations import CONFIGURATION, Loader
import torch, time

# --- IF SAM MODEL ---

if CONFIGURATION["MODEL_VERSION"] == "sam":
    !pip install git+https://github.com/facebookresearch/segment-anything.git
    from segment_anything import sam_model_registry                 # DOWNLOAD

# --- MOUNT DRIVE AND EXTRACT DATASET ---

drive.mount(CONFIGURATION["PATH_DRIVE"], force_remount=True)
!tar -xzf {CONFIGURATION["PATH_FILE"]}

# --- LOAD MODEL ---

print()
print("Loading ", CONFIGURATION["MODEL_VERSION"], " model...")

if CONFIGURATION["MODEL_VERSION"] == "dinov2":
    MODEL = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14_reg", weights=CONFIGURATION["PTH_PATH"])
elif CONFIGURATION["MODEL_VERSION"] == "dinov3":
    MODEL = torch.hub.load("facebookresearch/dinov3", "dinov3_vitb16", weights=CONFIGURATION["PTH_PATH"])
elif CONFIGURATION["MODEL_VERSION"] == "sam":
    MODEL = sam_model_registry["vit_b"](checkpoint=CONFIGURATION["PTH_PATH"])
MODEL = MODEL.to(CONFIGURATION["DEVICE"])

if CONFIGURATION["DEVICE"] == "cuda":
    torch.cuda.synchronize()                         # SYNCHROIZE GPU

    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)          # GO
    start_event.record()

else:
    start_event = time.time()       # GO

loader = Loader(0, batch_size=1)
run_evaluation(loader, "test")

if CONFIGURATION["DEVICE"] == "cuda":
    torch.cuda.synchronize()          # STOP
    end_event.record()
else:
    end_event = time.time()


elapsed_time = start_event.elapsed_time(end_event) / 1000              # SECONDS
print()
print("Analysis for: " + CONFIGURATION["DEVICE"])
print("Total required time: " + str(elapsed_time) + " seconds")

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-ozstsm14
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-ozstsm14
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
Mounted at /content/drive

Loading  sam  model...


Evaluating test PCK metrics:   0%|          | 0/2438 [00:00<?, ?it/s]

KeyboardInterrupt: 